# Chapter 27 — Replicate Before You Believe

**Companion to *Applied AI*.**

Chapter 26 fenced a signal: counterfactual wording, matched exposure, one
variable changed, rule frozen in advance. This notebook recomputes the matched
replication (P3, 288 calls) from its rows — and the replication ties. The
chapter's core lesson is executable:

## Question

**Did the promising signal survive a matched replication?**

## What this notebook does

It **reproduces** `p-series/p3/p3-export.json`: the tie (9/12 tasks and
44/144 candidates on both arms, at +45.8% tokens), the per-task pass table,
movement summing to 18, and the post-hoc permutation check at p ≈ 0.31 —
computed here, labeled as post-hoc, and kept unpromotable.

```text
failed replication  =  successful experiment
```

## Setup

Standard library only. No network, no API key, no `codeai` import. Only
bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
import random
from collections import defaultdict
from math import comb
from pathlib import Path

def find_evidence_dir(marker="p-series"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
EXPORT = EVIDENCE_DIR / "p-series" / "p3" / "p3-export.json"
print("bundle: p-series/p3/p3-export.json")
data = json.loads(EXPORT.read_text(encoding="utf-8"))
cands = data["candidates"]
calls = {c["call_id"]: c for c in data["calls"]}
print("arms: P3C = normal x12 | P3CF = counterfactual x12 (matched exposure)")
print("candidates:", len(cands))

bundle: p-series/p3/p3-export.json
arms: P3C = normal x12 | P3CF = counterfactual x12 (matched exposure)
candidates: 288


## 1. Discovery result, matched replication, difference, decision

The four columns of a replication, recomputed from the rows. Same tasks,
twelve draws per task on both sides, one variable changed:

In [2]:
PASS = "VERIFIER_PASS"
solved = defaultdict(set)
passes = defaultdict(int)
toks = defaultdict(int)
for c in cands:
    call = calls[c["call_id"]]
    toks[c["arm"]] += call["input_tokens"] + call["output_tokens"]
    if c["outcome"] == PASS:
        solved[c["arm"]].add(c["corpus_task_id"])
        passes[c["arm"]] += 1

print(f"P3C  normal        : {len(solved['P3C'])}/12 tasks, {passes['P3C']}/144 candidates")
print(f"P3CF counterfactual: {len(solved['P3CF'])}/12 tasks, {passes['P3CF']}/144 candidates")
print(f"token overhead: +{(toks['P3CF'] / toks['P3C'] - 1) * 100:.1f}%")
assert len(solved["P3C"]) == len(solved["P3CF"]) == 9
assert solved["P3C"] == solved["P3CF"]
assert (passes["P3C"], passes["P3CF"]) == (44, 44)
assert abs(toks["P3CF"] / toks["P3C"] - 1.458) < 0.005
print()
print("Decision: the frozen rule keeps the default. A tie at +45.8% tokens")
print("is not a win. The replication did its job by preventing a promotion.")

P3C  normal        : 9/12 tasks, 44/144 candidates
P3CF counterfactual: 9/12 tasks, 44/144 candidates
token overhead: +45.8%

Decision: the frozen rule keeps the default. A tie at +45.8% tokens
is not a win. The replication did its job by preventing a promotion.


## 2. The tie that moved

Equal aggregates do not mean identical rows. Per-task passes out of 12,
recomputed here and matching the report cell for cell — up six on one task,
down four on another:

In [3]:
per_task = defaultdict(lambda: defaultdict(int))
for c in cands:
    if c["outcome"] == PASS:
        per_task[c["corpus_task_id"]][c["arm"]] += 1

TASKS = sorted({c["corpus_task_id"] for c in cands})
rows = [(per_task[t]["P3C"], per_task[t]["P3CF"]) for t in TASKS]
print(f"{'task':<22}{'normal':<9}counterfactual")
print("-" * 42)
for t, (a, b) in zip(TASKS, rows):
    print(f"{t:<22}{a:<9}{b}")
assert sum(a for a, _ in rows) == sum(b for _, b in rows) == 44

movement = sum(abs(a - b) for a, b in rows)
print("total movement:", movement)
assert movement == 18
print()
print("Success sat in different places without improving the promotion")
print("metric. 'Nothing differed' is ruled out; wording-caused redistribution")
print("is not established. Unfold the aggregate before concluding nothing")
print("happened — then test the unfolded rows before concluding something did.")

task                  normal   counterfactual
------------------------------------------
v2-dt-roundtrip       10       6
v2-env-timing         10       8
v2-half-up-rounding   6        5
v2-lsp-square         6        8
v2-registry-pollution 1        1
v2-retry-once-accepted0        0
v2-single-append      1        2
v2-size-format        0        0
v2-splitlines-cr      3        2
v2-stable-priority    4        3
v2-strip-query        0        0
v2-unsound-cache      3        9
total movement: 18

Success sat in different places without improving the promotion
metric. 'Nothing differed' is ruled out; wording-caused redistribution
is not established. Unfold the aggregate before concluding nothing
happened — then test the unfolded rows before concluding something did.


## 3. Post-hoc diagnostic, labeled post-hoc

Nothing was preregistered to judge the movement, so this is a diagnostic,
not a verdict. Under an exchangeability null — each task's 24 draw outcomes
exchangeable between two groups of twelve — shuffle and ask how often total
movement reaches the observed 18. The exchangeability assumption is part of
the diagnostic; the rows do not independently establish it.

In [4]:
random.seed(20260914)

def shuffled_movement():
    total = 0
    for a, b in rows:
        draws = [1] * (a + b) + [0] * (24 - a - b)
        random.shuffle(draws)
        x = sum(draws[:12])
        total += abs(x - (a + b - x))
    return total

trials = 20_000
p = sum(shuffled_movement() >= movement for _ in range(trials)) / trials
print(f"observed movement: {movement} | shuffles reaching it: p ~= {p:.2f}")
assert 0.15 < p < 0.5   # ~= 0.31: chance-like allocation, not a stable effect
print()
print("About a third of shuffles move at least as much. The single largest")
print("shift (unsound-cache, 3 vs 9) has an uncorrected Fisher p of ~0.04 —")
print("across twelve inspected rows, a Bonferroni correction would not retain")
print("it. Task-by-prompt interaction noise, not a stable framing effect.")
print("Observed redistribution only: unpromotable.")

observed movement: 18 | shuffles reaching it: p ~= 0.32

About a third of shuffles move at least as much. The single largest
shift (unsound-cache, 3 vs 9) has an uncorrected Fisher p of ~0.04 —
across twelve inspected rows, a Bonferroni correction would not retain
it. Task-by-prompt interaction noise, not a stable framing effect.
Observed redistribution only: unpromotable.


## 4. Housekeeping the run honestly

288 calls produced 286 checks. The frozen rows identify both gaps — one
`EXECUTION_ERROR` candidate per arm with no check ID. Candidates that fail in
execution never reach the hidden tests, and the notebook counts them instead
of silently dropping them:

In [5]:
errs = [c for c in cands if c["outcome"] == "EXECUTION_ERROR"]
print("checks recorded:", len(data["checks"]), "| candidates:", len(cands))
print("execution-error candidates:", len(errs))
for c in errs:
    print(f"  arm={c['arm']} task={c['corpus_task_id']} check_id={c.get('check_id')}")
assert len(data["checks"]) == 286 and len(errs) == 2
print()
print("A failed replication is a successful experiment: it stopped an")
print("unsupported promotion, with the receipts intact.")

checks recorded: 286 | candidates: 288
execution-error candidates: 2
  arm=P3C task=v2-stable-priority check_id=None
  arm=P3CF task=v2-strip-query check_id=None

A failed replication is a successful experiment: it stopped an
unsupported promotion, with the receipts intact.


## Interpretation

1. **A signal ≠ a result ≠ a default.** The P2 signal met new data under a
   frozen rule and tied. The default stands.
2. **Cost belongs in the headline.** 44 = 44 reads as a tie; 44 = 44 at
   +45.8% tokens reads as a loss. The notebook keeps all three numbers
   together.
3. **Unfold ties before concluding.** Movement of 18 ruled out "nothing
   differed" — and the post-hoc check (p ≈ 0.31) ruled out promoting the
   difference.
4. **Post-hoc stays post-hoc.** The diagnostic was not preregistered, so it
   can only fence, never promote.

## Try it yourself

1. Rerun the permutation cell with a different seed. How much does p move —
   and does the qualitative reading survive?
2. Apply the Bonferroni correction to the unsound-cache Fisher p across
   twelve tasks. What remains?
3. Write the promotion rule P3 *would* have needed to pass (coverage,
   candidates, cost). Would you sign it before the next replication?

*Evidence: `experiments/applied-ai/evidence/p-series/p3/p3-export.json`
(288 calls, 286 checks, byte-pinned export). No network, no API key, no
`codeai` import.*